# 04 Root Finding

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/02-Numerical-Methods/04_Root_Finding.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=02-Numerical-Methods/04_Root_Finding.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)
## The Lens: Equilibrium is a Zero
**What problem are we solving?**  
In economics, equilibrium is defined by balance. 
*   **Market Clearing:** Supply equals Demand $\implies S(p) - D(p) = 0$.
*   **Arbitrage:** Returns are equalized $\implies R_A - R_B = 0$.
*   **Steady State:** Capital doesn't change $\implies s f(k) - \delta k = 0$.

These are all **root-finding problems**. We are looking for the variable $x$ such that $f(x) = 0$.

**Why this method?**  
Analytical solutions ($x = \frac{-b \pm \dots}{2a}$) are rare. We need numerical algorithms that can hunt down the zero for any function.
*   **Bisection:** Slow but guaranteed. Good for 1D problems where you know the bounds.
*   **Newton-Raphson:** Fast but risky. Requires derivatives.
*   **Brent's Method:** The industrial standard (hybrid). Safest for 1D.
*   **Homotopy Continuation:** A robust method for difficult systems of equations.

**Economic question.** In *04 Root Finding*, what must remain economically invariant when the computational representation changes? In economic computation, an algorithm is part of the model: convergence tolerances, conditioning, discretization, and stopping rules can alter the apparent equilibrium. The central question is therefore not merely whether a routine returns a number, but whether that number is stable under tighter tolerances, alternative initial conditions, and an independent method. Treat every numerical result as an approximation with a measurable error budget.

## Learning Objectives

By the end of this notebook, you will be able to:
1.  **Formulate** economic equilibrium problems as root-finding tasks.
2.  **Apply** the Fixed Point Iteration method and understand the Contraction Mapping Theorem.
3.  **Implement** Bisection and Newton's method.
4.  **Solve** for market clearing prices in a General Equilibrium model.
5.  **Utilize** Homotopy Continuation to solve systems where Newton's method fails.

## Prerequisites

*   **02-Numerical-Methods/03_Numerical_Differentiation.ipynb**: Newton's method relies on derivatives.


> **Learning path:** Building on [`03_Numerical_Differentiation.ipynb`](03_Numerical_Differentiation.ipynb); next continue with [`05_Optimization.ipynb`](05_Optimization.ipynb).


In [ ]:
# === Environment Setup ===
import warnings

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import brentq, newton

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'figure.dpi': 120})
%config InlineBackend.figure_format = 'retina'
np.set_printoptions(suppress=True, precision=4, linewidth=120)


## Interactive Lab: Market-Clearing Root

Use the demand intercept to shift the market and watch the numerical root move. The point is not the widget itself: it makes the mapping from $f(p)=Q_d(p)-Q_s(p)$ to an equilibrium price visible.


In [ ]:
try:
    from ipywidgets import FloatSlider, interact

    @interact(demand_intercept=FloatSlider(value=12.0, min=6.0, max=20.0, step=0.5, description="Demand a"))
    def market_clearing_widget(demand_intercept=12.0):
        demand_slope, supply_intercept, supply_slope = 1.2, 1.0, 0.8
        root = (demand_intercept - supply_intercept) / (demand_slope + supply_slope)
        residual = demand_intercept - demand_slope * root - (supply_intercept + supply_slope * root)
        print(f"Equilibrium price = {root:.3f}; excess-demand residual = {residual:.2e}")
except ImportError:
    print("Install ipywidgets to use the market-clearing slider.")


# Table of Contents
* [The Lens: Equilibrium is a Zero](#the-lens-equilibrium-is-a-zero)
* [Learning Objectives](#learning-objectives)
* [Prerequisites](#prerequisites)
* [1. Fixed-Point Theory](#1-fixed-point-theory)
  * [Visualizing Convergence: The Cobweb Plot](#visualizing-convergence-the-cobweb-plot)
* [2. Root-Finding Algorithms](#2-root-finding-algorithms)
  * [2.1 Bisection (Bracketing)](#21-bisection-bracketing)
  * [2.2 Newton-Raphson (Open)](#22-newton-raphson-open)
  * [Halley's Method (Cubic Convergence)](#halleys-method-cubic-convergence)
  * [Broyden's Method (Quasi-Newton)](#broydens-method-quasi-newton)
  * [Verbatim Walkthrough: Visualizing Newton's Failure](#verbatim-walkthrough-visualizing-newtons-failure)
  * [2.3 Brent's Method (The Best of Both)](#23-brents-method-the-best-of-both)
  * [Application: Yield to Maturity](#application-yield-to-maturity)
* [3. Systems of Equations: Market Equilibrium](#3-systems-of-equations-market-equilibrium)
* [4. Homotopy Continuation](#4-homotopy-continuation)
* [Summary](#summary)
* [Exercises](#exercises)
  * [1. Conceptual: Aitken's Acceleration](#1-conceptual-aitkens-acceleration)
  * [2. Applied: Implicit Yield Curve](#2-applied-implicit-yield-curve)
  * [3. Challenge: Multi-Market Equilibrium](#3-challenge-multi-market-equilibrium)


In [ ]:
# --- Anderson Acceleration (Full Implementation) ---
def anderson_acceleration(g, x0, m=5, tol=1e-6, max_iter=100):
    """Finds fixed point x = g(x) using Anderson mixing.
    
    Anderson Acceleration stores the last m iterates and their residuals,
    then solves a least-squares problem to find optimal mixing weights
    that minimize the predicted residual norm.
    
    Parameters
    ----------
    g : callable - The fixed-point map.
    x0 : float - Initial guess.
    m : int - Number of past iterates to mix (the "depth").
    tol : float - Convergence tolerance on |g(x) - x|.
    max_iter : int - Maximum iterations.
    
    Returns
    -------
    x : float - The approximate fixed point.
    k : int - Number of iterations used.
    """
    x = np.atleast_1d(np.asarray(x0, dtype=float))
    g_x = np.atleast_1d(g(x))
    res = g_x - x  # residual

    # Storage for past residuals and iterates
    X_hist = []  # past iterates
    F_hist = []  # past residuals

    for k in range(max_iter):
        if np.linalg.norm(res) < tol:
            return float(x) if x.size == 1 else x, k

        X_hist.append(x.copy())
        F_hist.append(res.copy())

        mk = min(len(X_hist) - 1, m)  # effective depth

        if mk == 0:
            # No history yet — plain Picard step
            x_new = g_x.copy()
        else:
            # Build the matrix of residual differences: dF[:,j] = F_{k} - F_{k-j}
            dF = np.column_stack([F_hist[-1] - F_hist[-1 - j] for j in range(1, mk + 1)])
            # Solve least-squares: min || F_k - dF @ gamma ||
            gamma, *_ = np.linalg.lstsq(dF, F_hist[-1], rcond=None)
            # Compute the Anderson update as a weighted combination
            dX = np.column_stack([X_hist[-1] - X_hist[-1 - j] for j in range(1, mk + 1)])
            x_new = g_x - (dF + dX) @ gamma

        x = x_new
        g_x = np.atleast_1d(g(x))
        res = g_x - x

    return float(x) if x.size == 1 else x, max_iter

# Note: scipy.optimize.fixed_point uses Anderson mixing by default via del2 acceleration
print("Anderson Acceleration is standard in solvers like scipy.optimize.root(method='krylov') for large systems.")


## 1. Fixed-Point Theory

Many economic problems are naturally phrased as finding a **fixed point**: $x = g(x)$.
*   Example: In the Solow model, $k_{t+1} = s k_t^\alpha + (1-\delta)k_t$. The steady state is $k^* = g(k^*)$.

**Fixed Point Iteration:** Guess $x_0$, then compute $x_{t+1} = g(x_t)$.

**Contraction Mapping Theorem (Banach):** If $g$ is a "contraction" (Lipschitz constant $\beta < 1$, i.e., slope strictly between -1 and 1 everywhere), this iteration is guaranteed to converge to a unique fixed point. This provides a *constructive* method to find the fixed point.

**Brouwer's Fixed Point Theorem:** A continuous function mapping a compact, convex set to itself has at least one fixed point. This guarantees *existence*, but not uniqueness, and doesn't tell us how to find it (non-constructive). It is crucial for proving existence of Nash Equilibria.

### Visualizing Convergence: The Cobweb Plot
For $x_{t+1} = g(x_t)$, we can visualize the path by drawing a line from $(x_t, x_t)$ vertically to the curve $(x_t, g(x_t))$, and then horizontally to $(g(x_t), g(x_t)) = (x_{t+1}, x_{t+1})$. This creates a "cobweb" or staircase pattern that spirals into the fixed point (if stable) or spirals out (if unstable).


## 2. Root-Finding Algorithms

Any fixed point problem $x = g(x)$ can be rewritten as a root finding problem $f(x) = x - g(x) = 0$.

### 2.1 Bisection (Bracketing)

If $f$ is continuous on $[a, b]$ with $f(a) < 0$ and $f(b) > 0$, the Intermediate Value Theorem guarantees a root in $(a, b)$. Bisection hunts it down by repeatedly halving the bracket:

1.  Check the midpoint $m = (a+b)/2$.
2.  If $f(m) > 0$, the root is in $[a, m]$. Else it is in $[m, b]$.
3.  Repeat with the new (half-width) bracket.

**Convergence rate — a complete derivation.** Let $[a_n, b_n]$ denote the bracket after $n$ halvings, and let $m_n = (a_n + b_n)/2$ be the estimate we report. Each step halves the bracket width exactly:

$$ b_n - a_n = \frac{b - a}{2^n}. $$

The root $x^*$ lies somewhere inside $[a_n, b_n]$, and $m_n$ sits at its center, so the error is at most half the bracket width:

$$ |m_n - x^*| \;\le\; \frac{b_n - a_n}{2} \;=\; \frac{b - a}{2^{\,n+1}}. $$

To guarantee $|m_n - x^*| \le \varepsilon$ we need $\frac{b-a}{2^{n+1}} \le \varepsilon$, i.e.

$$ n \;\ge\; \log_2\!\left(\frac{b - a}{\varepsilon}\right) - 1. $$

The error *bound* shrinks by the constant factor $\tfrac{1}{2}$ every iteration — this is **linear convergence with rate $\tfrac{1}{2}$**. Each step buys exactly one binary digit of accuracy, so roughly $\log_2 10 \approx 3.3$ iterations are needed per decimal digit. For $[a,b] = [0,1]$ and $\varepsilon = 10^{-8}$, that is $n \ge \log_2(10^8) - 1 \approx 26$ iterations — regardless of how nice or nasty $f$ is.

**Pros:** Guaranteed convergence; the error bound above depends only on the initial bracket. **Cons:** Slow (linear convergence), and it needs a sign-changing bracket to start.

### 2.2 Newton-Raphson (Open)

Use the tangent line to jump to the root:

$$ x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)} $$

**Derivation:** Newton's method comes from the first-order Taylor expansion $f(x) \approx f(x_n) + f'(x_n)(x - x_n)$. Setting this linear approximation to zero and solving for $x$ gives the update rule.

> **Theorem (Local Quadratic Convergence of Newton's Method):** Let $f$ be twice continuously differentiable near a root $x^*$ with $f'(x^*) \neq 0$. Then there exists $\delta > 0$ such that for any starting point $x_0$ with $|x_0 - x^*| \le \delta$, the Newton iterates converge to $x^*$ and satisfy
> $$ |e_{n+1}| \;\le\; M\, |e_n|^2, \qquad M = \frac{\sup_{|x - x^*| \le \delta} |f''(x)|}{2 \inf_{|x - x^*| \le \delta} |f'(x)|}, $$
> where $e_n = x_n - x^*$ is the error at step $n$.

**Proof.** Expand $f(x^*)$ around $x_n$ using Taylor's theorem with the Lagrange remainder:

$$ 0 = f(x^*) = f(x_n) + f'(x_n)(x^* - x_n) + \frac{f''(\xi_n)}{2}(x^* - x_n)^2, $$

where $\xi_n$ is some point strictly between $x_n$ and $x^*$ (its existence is what the Lagrange remainder guarantees). Divide through by $f'(x_n)$ (nonzero near $x^*$ by assumption) and use $x^* - x_n = -e_n$:

$$ 0 = \frac{f(x_n)}{f'(x_n)} - e_n + \frac{f''(\xi_n)}{2 f'(x_n)}\, e_n^2. $$

Newton's update gives $e_{n+1} = x_{n+1} - x^* = e_n - \frac{f(x_n)}{f'(x_n)}$, and substituting the display above for $e_n - f(x_n)/f'(x_n)$:

$$ e_{n+1} = \frac{f''(\xi_n)}{2 f'(x_n)}\, e_n^2. $$

Taking absolute values and bounding $|f''(\xi_n)|$ and $|f'(x_n)|$ over the $\delta$-neighborhood yields $|e_{n+1}| \le M |e_n|^2$. If additionally $M\delta < 1$, then $|e_1| \le M|e_0|^2 \le (M\delta)|e_0| < |e_0|$, so the iterates never leave the neighborhood and the bound applies at every step, giving convergence. $\blacksquare$

The error at the next step is proportional to the *square* of the current error: near the root, the number of correct digits roughly **doubles** each iteration.

**Pros:** Extremely fast (quadratic convergence). **Cons:** Can diverge if the initial guess is bad or the derivative is (near) zero — the theorem above is strictly *local*.

### Halley's Method (Cubic Convergence)

If we keep the second-order term of the Taylor expansion instead of discarding it, we get **Halley's method**. Write the expansion at the current iterate, with step $h = x - x_n$:

$$ 0 \approx f(x_n) + f'(x_n)\, h + \tfrac{1}{2} f''(x_n)\, h^2 . $$

This is quadratic in $h$; rather than solving it exactly (which needs a square root and a branch choice), factor out one power of $h$ and substitute Newton's step $h \approx -f(x_n)/f'(x_n)$ inside the correction term:

$$ h = \frac{-f(x_n)}{f'(x_n) + \tfrac{1}{2} f''(x_n)\, h} \;\approx\; \frac{-f(x_n)}{f'(x_n) - \dfrac{f(x_n) f''(x_n)}{2 f'(x_n)}}, $$

which after clearing the nested fraction gives the update

$$ x_{n+1} = x_n - \frac{2 f(x_n) f'(x_n)}{2 [f'(x_n)]^2 - f(x_n) f''(x_n)}. $$

It converges at a **cubic** rate (the number of correct digits roughly triples per iteration) but requires the second derivative $f''(x)$ — worthwhile only when $f''$ is cheap, as for the smooth pricing functions we meet in finance.

### Broyden's Method (Quasi-Newton)

For systems of equations, calculating the Jacobian $J$ and solving the linear system at every step is expensive ($O(N^3)$). Broyden's method approximates the Jacobian and updates it using a rank-1 formula (the "Secant equation" for matrices).

$$ J_{n+1} = J_n + \frac{(y_n - J_n s_n)s_n^T}{s_n^T s_n} $$

where $s_n = x_{n+1} - x_n$ and $y_n = F(x_{n+1}) - F(x_n)$. This allows for $O(N^2)$ updates. It converges superlinearly (slower than quadratic, but much cheaper per step).

**Acceleration Methods: Anderson Mixing**

For fixed-point problems $x = g(x)$, simple iteration can be slow. **Anderson Acceleration** uses a weighted average of the last $m$ iterates to minimize the residual $x - g(x)$. It acts like a Quasi-Newton method but doesn't require computing a Jacobian. It is highly effective in solving Dynamic Programming problems (Value Function Iteration).

**Basins of Attraction:** The set of initial guesses that converge to a specific root is called its basin of attraction. For complex functions, these basins can be fractal (e.g., the Newton fractal).

### Verbatim Walkthrough: Visualizing Newton's Failure

Newton's method is fast but fragile. If the function has a flat spot (derivative near zero), it sends the next guess to infinity. If it has cycles, it oscillates forever.

Let's visualize the "Basins of Attraction" for $z^3 - 1 = 0$ in the complex plane. Each color represents which of the 3 roots the method converges to. The boundary is fractal!


In [ ]:
def newton_fractal(img_size=200):
    # Create a grid of complex points
    x = np.linspace(-1.5, 1.5, img_size)
    y = np.linspace(-1.5, 1.5, img_size)
    X, Y = np.meshgrid(x, y)
    Z = X + 1j * Y

    # Roots of z^3 - 1 = 0 are 1, e^(2pi*i/3), e^(4pi*i/3)
    roots = np.roots([1, 0, 0, -1])

    # Iterate Newton's method: z_new = z - f(z)/f'(z)
    # f(z) = z^3 - 1, f'(z) = 3z^2
    for i in range(20):
        mask = Z != 0
        Z[mask] = Z[mask] - (Z[mask]**3 - 1) / (3 * Z[mask]**2)

    # Classify points based on which root they are close to
    colors = np.zeros(Z.shape)
    for i, r in enumerate(roots):
        dist = np.abs(Z - r)
        colors[dist < 1e-4] = i + 1

    plt.figure(figsize=(6, 6))
    plt.imshow(colors, extent=[-1.5, 1.5, -1.5, 1.5], cmap='viridis')
    plt.title("Newton Fractal: Where does your guess go?")
    plt.xlabel("Re(z)")
    plt.ylabel("Im(z)")
    plt.show()

newton_fractal()


### 2.3 Brent's Method (The Best of Both)

Bisection is safe but slow; Newton and the secant method are fast but can escape the bracket or diverge. **Brent's method** (Brent, 1973) is the industrial-strength hybrid that keeps the guarantee of one and the speed of the others. `scipy.optimize.brentq` implements it, and it is the right default for any scalar root-finding problem.

**How the hybridization actually works.** Brent maintains a sign-changing bracket $[a, b]$ at all times (like bisection) and, at each iteration, chooses among three candidate steps, from most to least ambitious:

1.  **Inverse quadratic interpolation (IQI).** With three points $(a, f(a))$, $(b, f(b))$, $(c, f(c))$ available, fit a *sideways* parabola $x = q(y)$ through them and evaluate it at $y = 0$. Interpolating $x$ as a function of $y$ — rather than $y$ as a function of $x$ — means "where the curve crosses zero" is read off directly, with no equation to solve:
    $$ x_{\text{new}} = \sum_{i \in \{a,b,c\}} x_i \prod_{j \neq i} \frac{-f(x_j)}{f(x_i) - f(x_j)} \qquad \text{(Lagrange form at } y = 0\text{)}. $$
2.  **Secant step.** If only two distinct function values are available (or the IQI denominators degenerate because two $f$ values coincide), fall back to the secant line through the two best points.
3.  **Bisection.** The safety net.

The interpolation step is **accepted only if it passes safeguards**; otherwise the iteration takes a bisection step instead. The two key conditions are:

- the candidate must land *inside* the current bracket (an interpolated point outside $[a,b]$ is meaningless as a root estimate); and
- it must make *sufficient progress* — roughly, each interpolation step must move less than half of the second-to-last step, so that two consecutive interpolations shrink the interval faster than bisection would. If steps stagnate, Brent forces a bisection.

**What this buys us.** Because a valid bracket is preserved at every iteration, Brent inherits bisection's worst-case guarantee (never worse than about $2\times$ the bisection iteration count). Because IQI/secant steps are taken whenever they behave, it inherits superlinear local convergence near a simple root (order $\approx 1.84$ for IQI, $\approx 1.62$ for secant) — all *without ever evaluating a derivative*.

**Rule of thumb:** for a scalar problem where you can bracket the root, use `brentq` and stop thinking about it. Reach for Newton/Halley only when derivatives are cheap and you need every last drop of speed (e.g., inside a hot loop repricing thousands of bonds), and for Broyden/Anderson when the problem is a *system*.


### Application: Yield to Maturity

The price of a bond is the present value of its payments. The Yield to Maturity (YTM) is the interest rate $y$ that makes the present value equal to the market price $P_{market}$.

$$ P(y) - P_{market} = 0 $$


In [ ]:
def bond_excess_price(y, price, coupon, face, T):
    # PV of coupons
    pv_coupons = sum(coupon / (1+y)**t for t in range(1, T+1))
    # PV of face value
    pv_face = face / (1+y)**T
    return (pv_coupons + pv_face) - price

# Bond Params
market_price = 950
coupon = 50
face = 1000
T = 10

# Find YTM using Brent's method
# We know yield must be between 0% and 20%
ytm = brentq(bond_excess_price, 0.0, 0.2, args=(market_price, coupon, face, T))

print(f"Market Price: ${market_price}")
print(f"Yield to Maturity: {ytm:.2%}")


## 3. Systems of Equations: Market Equilibrium

For $N$ markets, we have a system $F(\mathbf{p}) = \mathbf{0}$, where $\mathbf{p}$ is a vector of prices.

**Example: General Equilibrium with CES Utility**
Two agents (A, B), two goods (1, 2). 
We want to find the relative price $p_1$ (normalize $p_2=1$) such that Excess Demand for Good 1 is zero.


In [ ]:
# Agent A: Prefers Good 1
alpha_A, rho_A = 0.7, -1.0 # CES params
endow_A = np.array([4.0, 1.0])

# Agent B: Prefers Good 2
alpha_B, rho_B = 0.3, -1.0
endow_B = np.array([1.0, 4.0])

def get_demand(p1, income, alpha, rho):
    # Marshallian demand for Good 1 from CES FOCs
    # Derived from: Max (alpha*c1^rho + (1-alpha)c2^rho)^(1/rho)
    p2 = 1.0
    sigma = 1 / (1 - rho)
    term = (alpha / (1-alpha) * p2 / p1)**sigma
    c1 = income / (p1 + p2/term)
    return c1

def excess_demand(p1):
    p2 = 1.0
    # 1. Calculate Incomes
    inc_A = p1*endow_A[0] + p2*endow_A[1]
    inc_B = p1*endow_B[0] + p2*endow_B[1]

    # 2. Calculate Demands for Good 1
    c1_A = get_demand(p1, inc_A, alpha_A, rho_A)
    c1_B = get_demand(p1, inc_B, alpha_B, rho_B)

    # 3. Supply
    supply_1 = endow_A[0] + endow_B[0]

    return (c1_A + c1_B) - supply_1

# Solve for equilibrium price
p1_star = brentq(excess_demand, 0.1, 10.0)

print(f"Equilibrium Price p1: {p1_star:.4f}")
print(f"Excess Demand at p1*: {excess_demand(p1_star):.2e}")


## 4. Homotopy Continuation

Finding roots for systems $F(x)=0$ is hard. If your initial guess is far off, Newton's method explodes.

**The Idea:** Start with an easy problem $G(x)=0$ (where you know the solution) and slowly transform it into the hard problem $F(x)=0$.

$$ H(x, t) = (1-t)G(x) + tF(x) $$

1.  Start at $t=0$ with known solution $x_0$.
2.  Increase $t$ slightly (e.g., $t=0.1$). Use $x_0$ as guess to solve $H(x, 0.1)=0$.
3.  Repeat until $t=1$.

This "path-following" keeps you in the basin of attraction of the root.


In [ ]:
# Difficult problem: x^3 - 1 = 0 (finding real root x=1 is easy, but let's pretend it's hard)
# Let's assume we don't know x=1.
# Easy problem: x - 0.5 = 0 (Solution x=0.5)

def F(x): return x**3 - 1
def G(x): return x - 0.5

def H(x, t):
    return (1-t)*G(x) + t*F(x)

x_curr = 0.5 # Known solution for t=0
t_steps = np.linspace(0, 1, 11)

print("Homotopy Path:")
for t in t_steps:
    # Solve H(x, t) = 0 using previous x as guess
    # Use newton for the step
    x_curr = newton(lambda x: H(x, t), x0=x_curr)
    print(f"  t={t:.1f}, Root={x_curr:.4f}")

print(f"Final Solution for F(x)=0: {x_curr:.4f}")


## Key Equations

These relations are collected from the derivations above as a review map. Their assumptions and derivations remain part of the result; this box is not a substitute for them.

**1. Core relation**

$$b_n - a_n = \frac{b - a}{2^n}.$$

**2. Core relation**

$$|m_n - x^*| \;\le\; \frac{b_n - a_n}{2} \;=\; \frac{b - a}{2^{\,n+1}}.$$

**3. Core relation**

$$n \;\ge\; \log_2\!\left(\frac{b - a}{\varepsilon}\right) - 1.$$

**4. Core relation**

$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$


## Summary

**Key Takeaways:**
*   **Everything is a Root:** Equilibrium, steady states, and arbitrage conditions are all $f(x)=0$.
*   **Scalar? Use Brent:** `brentq` is the default choice for 1D problems.
*   **Vector? Use Newton:** For systems, use `scipy.optimize.root` (which uses Newton-Krylov methods).
*   **Fails? Use Homotopy:** If Newton diverges, try path-following from an easier problem.


## Exercises

### 1. Conceptual: Aitken's Acceleration
Fixed point iteration converges linearly. Aitken's $\Delta^2$ method accelerates this to quadratic. Research the formula and explain *why* it works (hint: it estimates the geometric decay of the error).

### 2. Applied: Implicit Yield Curve
Write a function `get_yield(price, coupon, face, T)` that handles an array of bonds. Use a loop with `brentq` to calculate the yield curve for a set of maturities: $T=[1, 2, 5, 10, 30]$. Plot the result.

### 3. Challenge: Multi-Market Equilibrium
Extend the CES code to 3 goods. You will now have two relative prices ($p_1, p_2$) to solve for. Use `scipy.optimize.root` to solve the system of 2 excess demand equations.


## References & Further Reading

- Judd, K. L. (1998). *Numerical Methods in Economics*. MIT Press.
- Miranda, M. J. & Fackler, P. L. (2002). *Applied Computational Economics and Finance*. MIT Press.
- Nocedal, J. & Wright, S. J. (2006). *Numerical Optimization* (2nd ed.). Springer.
